# Model 1: Inference-Only Baseline with Groq API

**Author:** Berkay Kaya (Team 11)  
**Course:** Applications of Data Science -- SBWL Data Science, WU Wien, SS26

## Overview
This notebook implements the inference-only baseline for Austrian tax law question answering in German. It uses `llama-3.3-70b-versatile` via the Groq API with zero-shot prompting, meaning that the model is not additionally trained on the course dataset. The goal of this setup is to provide a strong API-based baseline that can later be compared against the fine-tuned and RAG-based approaches.

## Architecture
1. **Dataset Loading:** Read the 643 questions from `dataset_clean.csv`  
2. **Prompting:** Combine each question with a fixed system prompt for Austrian tax law answering  
3. **API Inference:** Send each prompt to `llama-3.3-70b-versatile` via the Groq API  
4. **Checkpointing:** Save intermediate results regularly to avoid losing progress  
5. **Result Export:** Store the final answers in CSV format for evaluation  

## Requirements
- Python environment with `groq` and `pandas` installed  
- A valid Groq API key  
- Input dataset: `dataset_clean.csv`  
- Internet connection for API access  

## How to Reproduce
1. Install dependencies: `pip install groq pandas`  
2. Set your Groq API key in the configuration cell  
3. Run all cells in order  
4. Results are saved to `../results/model1_api_results.csv`  

## Cell 1 - Install Packages
This cell installs the required Python packages for the notebook: `groq` and `pandas`.

- `import sys` loads Python’s built-in `sys` module, which provides information about the current Python interpreter.
- `sys.executable` returns the exact path of the Python executable that is currently running the notebook.
- The `!` tells Jupyter to run the command as a shell command instead of normal Python code.
- `-m pip install groq pandas` installs the `groq` and `pandas` packages into the same Python environment used by the notebook.

This is important because notebooks sometimes run in a different environment than the system terminal. By using `sys.executable`, the installation is guaranteed to happen in the correct environment.

### Project Impact
This cell prepares the technical foundation of the whole pipeline. If these packages are missing or installed in the wrong environment, the project cannot access the Groq API or process the dataset correctly.


In [3]:
# Install dependencies into the active kernel environment
import sys
!{sys.executable} -m pip install groq pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


## Cell 2 — Import Libraries
This cell imports all libraries that are needed for API communication, data handling, waiting between requests, and file checks.

- `from groq import Groq` imports the `Groq` client class, which is used to send requests to the Groq API.
- `import pandas as pd` imports the `pandas` library for working with tabular data such as CSV files. The alias `pd` is the common shorthand.
- `import time` gives access to timing functions such as `time.sleep()`, which can be used to pause between API calls.
- `import os` provides file system utilities, for example checking whether a checkpoint file already exists.

Together, these imports cover the core tasks of the notebook: sending model requests, reading and writing CSV files, handling delays, and managing files safely.
### Project Impact
This cell brings in the tools needed for the project workflow. Without these imports, the notebook would not be able to read the tax-law dataset, call the model, or recover progress from saved files.


In [4]:
from groq import Groq
import pandas as pd
import time
import os

## Cell 3 — Configuration

This cell defines the main settings for the project, including API access, model choice, response behavior, and file locations.

- `API_KEY` stores the personal key that authorizes access to the Groq API.
- `MODEL_NAME` defines which language model will answer the questions. In this notebook, the selected model is `llama-3.3-70b-versatile`.
- `TEMPERATURE = 0.1` controls randomness. A low value makes the output more stable and deterministic, which is useful for legal questions.
- `MAX_TOKENS = 300` limits the maximum length of each generated answer.
- `INPUT_CSV` points to the dataset file that contains the questions.
- `OUTPUT_CSV` defines where the final answers will be saved.
- `CHECKPOINT_CSV` defines where intermediate progress will be stored in case the notebook stops unexpectedly.
- `client = Groq(api_key=API_KEY)` creates the Groq API client using the provided key.

### Project Impact
This cell acts as the control center of the notebook. Instead of hard-coding settings throughout the script, everything important is defined in one place.

It determines how the model behaves and where the data flows through the project. Small changes here can directly affect answer quality, reproducibility, and the reliability of the final submission.

In [ ]:
API_KEY = "INSERT-YOUR-GROQ-API-KEY-HERE"
MODEL_NAME = "llama-3.3-70b-versatile"
TEMPERATURE = 0.1
MAX_TOKENS = 300

INPUT_CSV = "../../dataset_clean.csv"
OUTPUT_CSV = "../results/model1_api_results.csv"
CHECKPOINT_CSV = "../results/model1_checkpoint.csv"  # saves progress every 50 questions

client = Groq(api_key=API_KEY)

## Cell 4 — System Prompt
This cell defines the system prompt that guides the model’s role, language, style, and constraints for every answer.

- A system prompt is an instruction that is sent to the model before the user’s actual question.
- It defines how the model should behave during the conversation.
- In this project, the prompt tells the model to act as an expert in Austrian tax law.
- It also tells the model to answer in German, stay precise, cite relevant legal paragraphs, and keep the answer short.
- Triple quotes (`""" ... """`) allow multi-line text in Python.

The system prompt is especially important in a legal setting because it reduces ambiguity and keeps the model focused on the required output format.

### Project Impact
This cell strongly influences answer style and consistency across all 643 questions. A well-designed prompt can improve legal precision and formatting, while a weak prompt can lead to vague, inconsistent, or non-compliant answers.


In [ ]:
SYSTEM_PROMPT = """Du bist ein Experte für österreichisches Steuerrecht.
Beantworte die folgende Frage präzise und zitiere die relevanten Gesetzesparagrafen (z.B. § 7 Abs. 1 KStG, § 4 EStG, § 1 UStG).
Antworte ausschließlich auf Deutsch.
Halte die Antwort unter 250 Wörtern und fokussiere dich auf die wesentlichen Rechtsnormen und deren Anwendung."""

## Cell 5 — Load Data
This cell loads the input dataset and prints a small preview to confirm that the data was read correctly.

- `pd.read_csv(INPUT_CSV)` reads the CSV file from the configured input path and stores it in a pandas DataFrame called `df`.
- `len(df)` returns the number of rows in the dataset, which corresponds to the number of questions.
- `df.head(3)` displays the first three rows of the DataFrame for a quick manual check.
- `f"Loaded {len(df)} questions"` is an f-string, meaning the value inside `{}` is inserted directly into the text.

This step is useful for validating that the dataset path is correct and that the expected columns and rows are available before inference begins.

### Project Impact
This cell is the entry point of the project data pipeline. If the dataset is loaded incorrectly, all following model outputs will also be wrong or incomplete.


In [5]:
# Load dataset
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} questions")
print(df.head(3))

Loaded 643 questions
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...


## Cell 6 — Load Checkpoint
This cell checks whether previous progress already exists and resumes the process from that point if possible.

- `os.path.exists(CHECKPOINT_CSV)` checks whether a checkpoint file is already available.
- If the file exists, `pd.read_csv(CHECKPOINT_CSV)` loads the saved progress.
- `done["id"].tolist()` extracts the finished question IDs as a Python list.
- `set(...)` converts that list into a set, which allows very fast membership checks.
- `done.to_dict("records")` converts the checkpoint table into a list of dictionaries, for example:
  `[{ "id": "...", "answer": "..." }, ...]`
- If no checkpoint exists, the code starts with an empty set of completed IDs and an empty answer list.

This logic prevents the notebook from losing all progress after a crash, timeout, or manual interruption.

### Project Impact
This cell makes the project much more robust and practical for large API runs. It saves time, avoids repeated costs, and protects the workflow from restarting from zero after an interruption.


In [6]:
# Resume from checkpoint if it exists
if os.path.exists(CHECKPOINT_CSV):
    done = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done["id"].tolist())
    print(f"Resuming from checkpoint: {len(done_ids)} already done")
    answers = done.to_dict("records")
else:
    done_ids = set()
    answers = []

Resuming from checkpoint: 643 already done


## Cell 7 — Main Inference Loop
This cell is the core of the notebook. It sends each question to the Groq API, handles retries, skips completed rows, and saves progress regularly.

- `for i, row in df.iterrows():` loops through the dataset row by row.
- `i` is the row index, and `row` contains the current row’s values.
- `if row["id"] in done_ids:` checks whether the current question has already been answered.
- `continue` skips the row if it is already finished.
- `for attempt in range(3):` allows up to three tries for each API request.
- `try:` starts a block that may fail safely.
- `client.chat.completions.create(...)` sends the system prompt and the current user prompt to the Groq model.
- `messages` contains the prompt sequence: first the system role, then the actual question.
- `response.choices[0].message.content.strip()` extracts the first returned answer and removes extra spaces at the beginning and end.
- `answers.append(...)` adds the result to the growing answer list.
- `break` exits the retry loop once a request succeeds.
- `except Exception as e:` catches any error and stores it in `e`.
- `time.sleep(2 ** attempt)` applies exponential backoff: 1 second, then 2 seconds, then 4 seconds.
- `if (i + 1) % 50 == 0:` saves a checkpoint every 50 processed rows.
- `pd.DataFrame(answers).to_csv(...)` writes the current progress to disk.
- `time.sleep(0.3)` adds a short pause after each request.

This cell combines iteration, API calling, error handling, checkpointing, and pacing into one inference pipeline.

### Project Impact
This cell produces the actual model answers that will be submitted for evaluation. It is the most important execution step because it turns the dataset into a complete result file while keeping the run stable and recoverable.


In [ ]:
# Run inference for all 643 questions
for i, row in df.iterrows():
    if row["id"] in done_ids:
        continue

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row["prompt"]}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            answer = response.choices[0].message.content.strip()
            answers.append({"id": row["id"], "answer": answer})
            break
        except Exception as e:
            print(f"Error on {row['id']} attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)

    # Save checkpoint every 50 questions
    if (i + 1) % 50 == 0:
        pd.DataFrame(answers).to_csv(CHECKPOINT_CSV, index=False)
        print(f"Progress: {len(answers)}/{len(df)}")

    time.sleep(0.3)  # rate limit buffer

print(f"Done: {len(answers)} answers collected")

## Cell 8 — Save Results
This cell converts the collected answers into a final DataFrame, restores the original row order, creates the output folder if needed, and writes the final CSV file.

- `pd.DataFrame(answers)` converts the list of answer dictionaries into a pandas DataFrame.
- `id_order = df["id"].tolist()` stores the original order of IDs from the input dataset.
- `set_index("id")` temporarily uses the `id` column as the DataFrame index.
- `.loc[id_order]` reorders the results to match the exact order of the input file.
- `.reset_index()` turns the `id` back into a normal column.
- `os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)` creates the output directory if it does not already exist.
- `exist_ok=True` prevents an error if the folder is already present.
- `result_df.to_csv(OUTPUT_CSV, index=False)` saves the final results without adding an extra pandas index column.

Reordering is important because answers may have been collected over multiple runs or resumed from a checkpoint.

### Project Impact
This cell creates the final submission file in the expected structure. It ensures that the answers are saved cleanly and in the same order as the original dataset, which is essential for correct evaluation.


In [8]:
# Save final output
result_df = pd.DataFrame(answers)

# Reorder to match original input order
id_order = df["id"].tolist()
result_df = result_df.set_index("id").loc[id_order].reset_index()

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to {OUTPUT_CSV}")
print(result_df.head(3))

Saved to ../results/model1_api_results.csv
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Die Vergabe eines zinslosen Darlehens an einen...
2  CORP-TAX-003  Laut § 7 Abs. 1 Körperschaftsteuergesetz (KStG...


## Cell 9 — Verification
This cell validates the final output file and checks whether it meets the expected project format and completeness requirements.

- `pd.read_csv(OUTPUT_CSV)` loads the saved result file again for checking.
- `expected = len(df)` stores the expected number of rows based on the input dataset.
- `assert len(result) == expected` verifies that the output contains the correct number of rows.
- `assert list(result.columns) == ["id", "answer"]` checks that the output has exactly the required columns.
- `result["answer"].notna().all()` verifies that no answer is missing.
- `result["id"].nunique()` counts the number of unique IDs to detect duplicates.
- `list(result["id"]) == list(df["id"])` checks whether the result file preserves the exact original order of IDs.
- `assert` stops execution immediately if one of these conditions fails and shows the given error message.

This cell acts as a final quality-control step before submission.

### Project Impact
This cell reduces the risk of submitting a broken or invalid CSV file. It helps catch missing answers, duplicates, wrong columns, or ordering issues before the project is evaluated.


In [9]:
# Verification
result = pd.read_csv(OUTPUT_CSV)
expected = len(df)  # use actual dataset size, not hardcoded 644
assert len(result) == expected, f"Expected {expected} rows, got {len(result)}"
assert list(result.columns) == ["id", "answer"], f"Wrong columns: {list(result.columns)}"
assert result["answer"].notna().all(), "Some answers are null"
assert result["id"].nunique() == expected, "Duplicate IDs found"
assert list(result["id"]) == list(df["id"]), "ID order mismatch"
print(f"Verification passed. {expected} answers ready.")

Verification passed. 643 answers ready.
